# M1 experiment for determination of chunking size

## 1. Setup: load Doc01 paragraphs

In [1]:
import re
from m1_1_chunking import parse_paragraphs, DOC_PATH

paragraphs = parse_paragraphs(DOC_PATH)
long_paragraphs = [p["text"] for p in paragraphs if len(p["text"]) > 1000]
print(len(paragraphs), "paragraphs,", len(long_paragraphs), "over 1,000 characters:", [len(t) for t in long_paragraphs])

256 paragraphs, 10 over 1,000 characters: [1375, 1046, 1111, 1350, 1513, 3915, 1352, 1114, 1325, 1080]


## 2. Sentence splitter
Split after `.`, `!` or `?` when the next word starts with a capital letter, a digit, a quote or a bracket.

In [2]:
SENTENCE_END = re.compile(r"(?<=[.!?])\s+(?=[A-ZÇĞİÖŞÜ0-9“\"'(])")

def split_sentences(text):
    return SENTENCE_END.split(text)

sentences = [s for p in paragraphs for s in split_sentences(p["text"])]
lengths = sorted(len(s) for s in sentences)
print(len(sentences), "sentences; median", lengths[len(lengths) // 2], "longest", lengths[-1])

544 sentences; median 167 longest 563


In [3]:
# Check for false splits: very short "sentences" and dots inside sentences (abbreviations like "ör.", "vb.")
print("Shortest sentences:", [s for s in sentences if len(s) < 25])
print("Short word + dot inside the text:", sorted(set(re.findall(r"\b\w{1,4}\.(?=\s)", " ".join(p["text"] for p in paragraphs)))))

Shortest sentences: ['Kamu alımları bu şekilde', 'Eylemlerin göstergeleri:']
Short word + dot inside the text: ['4.', 'alan.', 'alır.', 'eder.', 'seti.', 'yapı.', 'yolu.', 'ör.']


## 3. Packing function

In [4]:
def split_long_sentence(sentence, max_chars):
    """Fallback: cut a sentence longer than max_chars at the last space before the limit."""
    pieces = []
    while len(sentence) > max_chars:
        cut = sentence.rfind(" ", 0, max_chars + 1)
        if cut <= 0:                      # no space at all: hard cut
            cut = max_chars
        pieces.append(sentence[:cut])
        sentence = sentence[cut:].lstrip()
    pieces.append(sentence)
    return pieces


def pack_paragraph(text, max_chars=1000, overlap=False):
    """Fill chunks with whole sentences up to max_chars; optionally repeat the last sentence in the next chunk."""
    units = [piece for s in split_sentences(text) for piece in split_long_sentence(s, max_chars)]
    chunks, current = [], []
    for unit in units:
        if current and len(" ".join(current + [unit])) > max_chars:
            chunks.append(" ".join(current))
            # overlap only if the repeated sentence and the next unit still fit together
            if overlap and len(current[-1]) + 1 + len(unit) <= max_chars:
                current = [current[-1], unit]
            else:
                current = [unit]
        else:
            current.append(unit)
    chunks.append(" ".join(current))
    return chunks


longest = max(long_paragraphs, key=len)
print("Longest paragraph:", len(longest), "characters,", len(split_sentences(longest)), "sentences")
print("Chunk lengths:", [len(c) for c in pack_paragraph(longest, 1000)])

Longest paragraph: 3915 characters, 19 sentences
Chunk lengths: [789, 986, 921, 860, 355]


## 4. Fallback test
Doc01 has no sentence over 1,000 characters, so the fallback is forced in two ways: a low `max_chars` on real text, and an artificial long sentence.

In [5]:
def check_fallback(text, max_chars):
    chunks = pack_paragraph(text, max_chars)
    return {
        "chunks": len(chunks),
        "longest_chunk": max(len(c) for c in chunks),
        "all_within_limit": all(len(c) <= max_chars for c in chunks),
        # paragraph text has single spaces, so this is only True if every cut fell at a space
        "no_text_lost_and_cuts_at_spaces": " ".join(chunks) == text,
    }

# (a) real paragraph, low limit
print("Real paragraph, max 300:", check_fallback(longest, 300))

# (b) artificial sentence over 1,000 characters (one real sentence repeated, no sentence end inside)
base = split_sentences(longest)[0].rstrip(".")
artificial = ", ".join([base] * 8) + "."
print("Artificial sentence length:", len(artificial))
print("Artificial, max 1000:", check_fallback(artificial, 1000))

Real paragraph, max 300: {'chunks': 18, 'longest_chunk': 298, 'all_within_limit': True, 'no_text_lost_and_cuts_at_spaces': True}
Artificial sentence length: 1479
Artificial, max 1000: {'chunks': 2, 'longest_chunk': 995, 'all_within_limit': True, 'no_text_lost_and_cuts_at_spaces': True}


## 5. Overlap test

In [6]:
def check_overlap(text, max_chars):
    chunks = pack_paragraph(text, max_chars, overlap=True)
    plain = pack_paragraph(text, max_chars, overlap=False)
    units = [piece for s in split_sentences(text) for piece in split_long_sentence(s, max_chars)]
    with_overlap = skipped = 0
    for prev, nxt in zip(chunks, chunks[1:]):
        last_unit = next(u for u in reversed(units) if prev.endswith(u))
        if nxt.startswith(last_unit):
            with_overlap += 1
        else:
            skipped += 1                  # repeated sentence + next sentence would not fit
    return {
        "chunks_without_overlap": len(plain),
        "chunks_with_overlap": len(chunks),
        "boundaries_with_overlap": with_overlap,
        "boundaries_skipped": skipped,
        "all_within_limit": all(len(c) <= max_chars for c in chunks),
        "extra_characters": sum(map(len, chunks)) - sum(map(len, plain)),
    }

print("Longest paragraph, max 1000:", check_overlap(longest, 1000))
print("Longest paragraph, max 300: ", check_overlap(longest, 300))

Longest paragraph, max 1000: {'chunks_without_overlap': 5, 'chunks_with_overlap': 6, 'boundaries_with_overlap': 5, 'boundaries_skipped': 0, 'all_within_limit': True, 'extra_characters': 906}
Longest paragraph, max 300:  {'chunks_without_overlap': 18, 'chunks_with_overlap': 18, 'boundaries_with_overlap': 2, 'boundaries_skipped': 15, 'all_within_limit': True, 'extra_characters': 250}


## 6. All long paragraphs, three settings

In [7]:
settings = [(1000, False), (1000, True), (300, False)]
for max_chars, overlap in settings:
    chunks = [c for t in long_paragraphs for c in pack_paragraph(t, max_chars, overlap)]
    fallback_cuts = sum(len(split_long_sentence(s, max_chars)) - 1 for t in long_paragraphs for s in split_sentences(t))
    lengths = [len(c) for c in chunks]
    print(f"max={max_chars:4} overlap={str(overlap):5}  chunks={len(chunks):3}  "
          f"shortest={min(lengths):4}  longest={max(lengths):4}  fallback_cuts={fallback_cuts}")

max=1000 overlap=False  chunks= 23  shortest= 165  longest= 986  fallback_cuts=0
max=1000 overlap=True   chunks= 24  shortest= 381  longest= 985  fallback_cuts=0
max= 300 overlap=False  chunks= 73  shortest=  27  longest= 300  fallback_cuts=9


## 7. Conclusion

Fallback is not neeeded for Doc01
Overlap 
- Is the fallback needed for Doc01?
- Does overlap help, or only add duplicated text?
- Which setting goes into `m1_1_chunking.py`?